In [ ]:
from game import Connect4
import numpy as np
import math
import time

import torch
print(torch.__version__)

torch.manual_seed(0) #for reproducability

import torch.nn as nn
import torch.nn.functional as F

2.13.0


## The Model

In [ ]:
class ResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden):
        super().__init__()
        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1), # 3 is the number of input planes that feed in
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for _ in range(num_resBlocks)]
        )

        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(), # why?
            nn.Linear(32 * game.row_count * game.col_count, game.row_count + 1), #final row count + 1 is the "action size"
        )

        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.row_count * game.col_count,  1),
            nn.Tanh()
        )

    
    def forward(self, x):
        x = self.startBlock(x)
        
        for resBlock in self.backBone:
            x = resBlock(x)

        policy = self.policyHead(x)
        value = self.valueHead(x)

        return policy, value

        
class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))

        x = x + residual
        x = F.relu(x) #why F.relu and nn.relu?

        return x
    
     

## Test the Model

In [9]:
game = Connect4()
model = ResNet(game, 4, 64)
state = game.get_initial_state()

state = game.make_move(state, 1, 0)
state = game.make_move(state, -1, 3)
state = game.make_move(state, 1, 2)
state = game.make_move(state, -1, 3)

encoded_state = game.get_encoded_state(state)
tensor_state = torch.tensor(encoded_state).unsqueeze(0)

policy, value = model(tensor_state)
value = value.item()
policy = torch.softmax(policy, axis = 1).squeeze(0).detach().cpu().numpy()

print(value, policy)

0.3501523733139038 [0.09907905 0.24637353 0.160626   0.1395051  0.10812383 0.14582118
 0.10047127]


In [ ]:
def sample_move(valid_moves):
    chosen_move = np.random.choice(np.flatnonzero(valid_moves))
    #print(f"playing move {chosen_move}")
    return chosen_move

class Node:
    def __init__(self, args, game, state, player, action_taken=None, parent = None, prior=0) -> None:

        self.game = game
        self.args = args
        self.state = state.copy()
        self.player = player

        self.visit_count = 0
        self.win_count = 0
        self.parent = parent
        self.children = []
        self.action_taken = action_taken
        self.prior = prior #this is for the UCB function in the alpha alg


    def visit(self):
        self.visit_count += 1

    def is_terminal(self):
        if not self.action_taken:
            return False
        
        _, terminated = self.game.get_value_and_terminated(self.state, self.action_taken)
        return terminated

    def is_fully_expanded(self):
        return len(self.children) > 0

    def has_children(self):
        return len(self.children) > 0

    def best_child(self): 
        
        if len(self.children) == 0:
            raise "no children to explore!"
        
        for child in self.children:
            if child.visit_count == 0:
                return child

        def ucb(child):
            if child.visit_count == 0:
                q_value = 0
            else:
                q_value = 1 - ((child.value_sum / child.visit_count)+ 1) / 2

            return  q_value + self.args['c'] * (math.sqrt(self.visit_count) / (child.visit_count + 1)) * child.prior 

        return max(self.children, key=ucb)
    
    def expand(self, policy):
        if self.is_fully_expanded():
            raise

        for action, prob in enumerate(policy):
            if prob > 0:
                child_state = self.game.make_move(self.state, self.player, action)
            
                child_node = Node(
                    args = self.args, 
                    game = self.game,
                    state = child_state,
                    player = self.game.get_opponent(self.player),
                    action_taken=action,
                    parent=self,
                    prior=prob
                )

                self.children.append(child_node)

    
    def rollout(self): # dead code now that we use the model
        
        state = self.state
        action_taken = self.action_taken
        player = self.player

        while True:
            value, terminated = self.game.get_value_and_terminated(state, action_taken)
            if terminated:
                #print(f"game finished, returning {value * player * -1}")
                return value * player * -1

            valid_moves = self.game.get_valid_moves(state)
            action_taken = sample_move(valid_moves)

            state = self.game.make_move(state, player, action_taken)
            player = self.game.get_opponent(player)

            #self.game.print_board(state)


    def backpropogate(self, result):
        self.win_count += result
        self.visit()

        if self.parent is not None:
            self.parent.backpropogate(-1 * result)
        

In [ ]:
class MCTS:
    def __init__(self, game, args, model) -> None:
        self.game = game
        self.args = args
        self.model = model

    @torch.no_grad() #why? what does this do?
    def search(self, state, verbose = False):
       
        # define root
        root = Node(self.args, self.game, state, player=1)

        for i in range(self.args['iterations']):
            current_node = root
            
            while not current_node.is_terminal() and current_node.is_fully_expanded():
                current_node = current_node.best_child()
                if verbose:
                    self.game.print_board(current_node.state)
            
            if not current_node.is_terminal():
                policy, value = self.model(
                    torch.tensor(self.game.get_encoded_state(current_node.state)).unsqueeze(0)
                )
                policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()
                valid_moves = self.game.get_valid_moves(current_node.state)
                policy *= valid_moves
                policy /= np.sum(policy)

                value = value.item()

                current_node.expand(policy)
                    
            current_node.backpropogate(value)
            if verbose:
                print(f"iteration {i} finished, result = {result}")
        

        visits = np.zeros(self.game.col_count)
        for child in root.children:
            visits[child.action_taken] = child.visit_count

        return visits


In [ ]:
game = Connect4()
model = ResNet(game, 4, 64)
model.eval() #what does this do?

state_init = game.get_initial_state()

round_1 = MCTS(game=game, args={'iterations' : 2000, 'c' : 2}, model)
results = round_1.search(state=state_init, verbose=False)

In [ ]:
results

In [ ]:
connect4 = Connect4()
state = connect4.get_initial_state()
mcts = MCTS(game=connect4, args={'iterations' : 8000, 'c' : 1.41})


player = 1 
while True:
    if player == -1:
        legal_moves = connect4.get_valid_moves(state)
        print(f"Legal Moves: {[i for i in range(connect4.col_count) if legal_moves[i]]}")
        
        action = int(input(f"player {player}: "))
        if action < 0 or action >= connect4.col_count or not legal_moves[action]:
            print('illegal move')
            continue

    else:
        search = mcts.search(state=state)
        action = np.argmax(search)

    state = connect4.make_move(state, player, action)
    print(f"player {player} plays {action}")
    connect4.print_board(state)

    value, terminated = connect4.get_value_and_terminated(state, action)

    if terminated:
        if value == 1:
            print(f"Player {player} wins!")
        else:
            print("draw")
        break


    player = connect4.get_opponent(player)
